# Using rio-tiler in Lonboard

This notebook shows how to use `rio-tiler` as a dynamic tile source for a [Lonboard](https://developmentseed.org/lonboard/) `RasterLayer`.

Instead of pre-generating tiles, Lonboard calls a user-provided `fetch_tile(x, y, z)` coroutine whenever the map needs a tile. We implement that function with `rio-tiler` in two ways:

1. Using the **experimental async reader** (`rio_tiler.experimental.geotiff.Reader`), backed by [`async-geotiff`](https://github.com/developmentseed/async-geotiff) and [`obstore`](https://github.com/developmentseed/obstore).
2. Using the classic **rasterio-based `Reader`**, run in a worker thread with `asyncio.to_thread` so it doesn't block the event loop.

In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "lonboard>=0.17.0",
#     "rio-tiler",
#     "async-geotiff",
#     "obstore",
# ]
# ///

## Dependencies

Install [`uv`](https://docs.astral.sh/uv) and then launch this notebook with:

```
uvx juv run docs/src/examples/Using-rio-tiler-in-lonboard.ipynb
```

(The `uvx` command is included when installing `uv`).

In [ ]:
import asyncio

# Tile Matrix Set definitions (WebMercatorQuad, ...)
import morecantile
from morecantile.models import TMSBoundingBox

# rio-tiler: output data model and the experimental async GeoTIFF reader
from rio_tiler.models import ImageData
from rio_tiler.experimental.geotiff import Reader as AsyncReader

# Lonboard: map widget, raster tile layer and helpers
from lonboard import Map, RasterLayer
from lonboard._geoarrow.ops import Bbox
from lonboard.raster import EncodedImage

# Async GeoTIFF/COG reader and the object store used to fetch bytes over HTTP
from async_geotiff import GeoTIFF
from obstore.store import HTTPStore

## Open the COG

We use a Sentinel-2 True Color Image (TCI) Cloud Optimized GeoTIFF hosted on S3.

The file is opened **once** with `async_geotiff.GeoTIFF.open` (this reads the header and IFDs), and the resulting `GeoTIFF` object is reused for every tile request. We also grab the metadata Lonboard needs to set up the layer: geographic bounds, zoom range and center.

In [ ]:
# HTTP store pointing to the "directory" holding the COGs
store = HTTPStore("https://titiler-benchmark-public.s3.us-east-1.amazonaws.com/cogs")

# Open the GeoTIFF once (reads header + IFDs); the object is reused for all tiles
geotiff = await GeoTIFF.open("S2C_33TVF_20260928_0_L2A_TCI.tif", store=store)

async with AsyncReader(input=geotiff) as src:
    # Dataset bounds in WGS84 (west, south, east, north)
    bounds = src.get_geographic_bounds(crs="EPSG:4326")
    # Zoom levels (in WebMercatorQuad) matching the dataset's overviews/resolution
    minzoom = src.minzoom
    maxzoom = src.maxzoom
    # Initial map center
    center = ((bounds[0] + bounds[2]) / 2, (bounds[1] + bounds[3]) / 2)

## Tile rendering

Lonboard separates *fetching* tile data from *rendering* it. The render function receives whatever `fetch_tile` returned (here a rio-tiler `ImageData`) and must return an encoded image.

The TCI file is already 8-bit RGB, so no rescaling or colormap is needed: `ImageData.render` encodes the data (plus the mask as alpha channel) to PNG.

In [ ]:
def render_rgb_tile(img: ImageData) -> EncodedImage:
    """Convert the image data to an RGBA PNG."""
    # `render` returns PNG bytes; the mask is written as the alpha band
    return EncodedImage(data=img.render(img_format="PNG"), media_type="image/png")

## Option 1: Async reader

`fetch_tile` is a coroutine called by Lonboard for each visible tile. With the experimental `AsyncReader`, the tile read is natively async: the byte-range requests are awaited, so many tiles can be fetched concurrently without threads.

We also build a `WebMercatorQuad` TileMatrixSet whose `boundingBox` is restricted to the dataset extent, so Lonboard only requests tiles that intersect the image.

In [ ]:
async def fetch_tile(
    x: int,
    y: int,
    z: int,
) -> ImageData:
    """Read a WebMercatorQuad tile from the COG using the async reader."""
    # Reuse the already opened `geotiff` object (no need to re-read the header)
    async with AsyncReader(input=geotiff) as src:
        img = await src.tile(x, y, z)
    return img


# Restrict the WebMercatorQuad TMS to the dataset extent so Lonboard
# doesn't request tiles outside the image
west, south, east, north = bounds
web_mercator = morecantile.tms.get("WebMercatorQuad")
tms = web_mercator.model_copy(
    update={
        "boundingBox": TMSBoundingBox(
            lowerLeft=web_mercator.xy(west, south),
            upperRight=web_mercator.xy(east, north),
            crs=web_mercator.crs,
        ),
    },
)

layer = RasterLayer(
    _tile_matrix_set=tms,  # tiling grid used to compute (x, y, z)
    _crs="EPSG:3857",  # CRS of the returned tiles
    _fetch_tile=fetch_tile,  # async function returning tile data
    _render_tile=render_rgb_tile,  # function encoding tile data to an image
    min_zoom=0,
    max_zoom=maxzoom,  # don't request tiles finer than the native resolution
    extent=bounds,
    _bounds=Bbox(*bounds),
    _center=center,
)

# Create the map
m = Map(layer, height=800)
m

## Option 2: rasterio-based Reader

The same can be done with the classic `rio_tiler.io.Reader` (GDAL/rasterio). Its methods are synchronous, so we run them in a worker thread with `asyncio.to_thread` to avoid blocking the event loop (and the map) while tiles are being read.

Note: the dataset is opened for every tile request here, whereas the async version reuses the same `GeoTIFF` object.

In [ ]:
from rio_tiler.io import Reader

# Full URL to the same COG (GDAL handles HTTP access via /vsicurl/)
src_path = "https://titiler-benchmark-public.s3.us-east-1.amazonaws.com/cogs/S2C_33TVF_20260928_0_L2A_TCI.tif"


async def fetch_tile(
    x: int,
    y: int,
    z: int,
) -> ImageData:
    """Read a WebMercatorQuad tile with the rasterio Reader in a worker thread."""

    def get_tile(x, y, z):
        # Blocking (GDAL) read
        with Reader(input=src_path, options={}) as src:
            return src.tile(x, y, z)

    # Offload the blocking call so the event loop stays responsive
    img = await asyncio.to_thread(get_tile, x, y, z)
    return img


# Same TMS restricted to the dataset extent as above
west, south, east, north = bounds
web_mercator = morecantile.tms.get("WebMercatorQuad")
tms = web_mercator.model_copy(
    update={
        "boundingBox": TMSBoundingBox(
            lowerLeft=web_mercator.xy(west, south),
            upperRight=web_mercator.xy(east, north),
            crs=web_mercator.crs,
        ),
    },
)

layer = RasterLayer(
    _tile_matrix_set=tms,
    _crs="EPSG:3857",
    _fetch_tile=fetch_tile,
    _render_tile=render_rgb_tile,
    min_zoom=0,
    max_zoom=maxzoom,
    extent=bounds,
    _bounds=Bbox(*bounds),
    _center=center,
)

# Create the map
m = Map(layer, height=800)
m